# Tracklet Video Processing - Incremental Global ID + Query Sandbox

Standalone notebook for Kaggle.

- No imports from the current repo.
- All helper logic is defined as new functions in this notebook.
- Accuracy is prioritized over raw speed.
- Only the new 10-minute batch is processed.
- Old raw videos can be deleted after the batch finishes because the notebook persists:
  - batch-level unified outputs
  - partitioned history by date
  - an online gallery used for the next batch
  - query artifacts for later retrieval experiments

This notebook now includes both query directions so you can test them in one place:

1. `ReID vector retrieval` using the tracklet embeddings.
2. `Qwen text description + metadata schema` generated from representative crops.
3. `Hybrid query` that combines structured filters, text similarity, and optional anchor identity.


In [ ]:
# Kaggle usually already has torch. These installs make the notebook portable.
%pip install -q faiss-cpu torchreid opencv-python-headless pyarrow pillow transformers accelerate sentence-transformers


In [ ]:
import ast
import json
import re
from datetime import UTC, datetime
from pathlib import Path

import cv2
import faiss
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torchreid.utils import FeatureExtractor

CONFIG = {
    "tracking_input_dir": "/kaggle/input/tracking-batch",
    "video_root_dir": "/kaggle/input/video-batch",
    "working_dir": "/kaggle/working/tracklet_processing",
    "history_dir": "/kaggle/working/tracklet_processing/history",
    "online_gallery_dir": "/kaggle/working/tracklet_processing/online_gallery",
    "batch_output_dir": "/kaggle/working/tracklet_processing/batch_outputs",
    "query_artifact_dir": "/kaggle/working/tracklet_processing/query_artifacts",
    "reid_model_name": "osnet_x1_0",
    "embedding_dim": 512,
    "similarity_threshold": 0.72,
    "samples_per_tracklet": 8,
    "min_frames_per_tracklet": 3,
    "min_bbox_area": 1024,
    "online_gallery_hours": 6,
    "query_top_k": 10,
    "enable_qwen_annotation": False,
    "qwen_max_tracklets": 100,
    "qwen_vlm_model_name": "Qwen/Qwen2.5-VL-3B-Instruct",
    "text_embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

for folder_key in ["working_dir", "history_dir", "online_gallery_dir", "batch_output_dir", "query_artifact_dir"]:
    Path(CONFIG[folder_key]).mkdir(parents=True, exist_ok=True)

BATCH_TIMESTAMP_UTC = datetime.now(UTC).replace(microsecond=0)
BATCH_ID = BATCH_TIMESTAMP_UTC.strftime("%Y%m%dT%H%M%SZ")

print("device:", CONFIG["device"])
print("batch_id:", BATCH_ID)
print("enable_qwen_annotation:", CONFIG["enable_qwen_annotation"])


In [ ]:
def find_first_column(columns, candidates):
    lookup = {str(column).strip().lower(): column for column in columns}
    for name in candidates:
        column = lookup.get(str(name).strip().lower())
        if column is not None:
            return column
    return None


def parse_bbox_value(raw_value):
    if isinstance(raw_value, (list, tuple)) and len(raw_value) >= 4:
        values = list(raw_value[:4])
    elif isinstance(raw_value, str) and raw_value.strip():
        text = raw_value.strip()
        try:
            parsed = json.loads(text)
        except Exception:
            parsed = ast.literal_eval(text)
        if not isinstance(parsed, (list, tuple)) or len(parsed) < 4:
            return None
        values = list(parsed[:4])
    else:
        return None

    try:
        x1, y1, x2, y2 = [int(float(value)) for value in values]
    except Exception:
        return None

    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def load_rows_from_file(file_path):
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".parquet":
        return pd.read_parquet(file_path)
    if suffix == ".jsonl":
        return pd.read_json(file_path, lines=True)
    if suffix == ".json":
        raw_data = json.loads(file_path.read_text(encoding="utf-8"))
        if isinstance(raw_data, list):
            return pd.DataFrame(raw_data)
        for key in ["items", "tracklets", "detections", "candidates", "rows"]:
            value = raw_data.get(key)
            if isinstance(value, list):
                return pd.DataFrame(value)
        return pd.DataFrame([raw_data])
    raise ValueError(f"Unsupported input file: {file_path}")


def normalize_detection_frame(raw_df, source_path, config):
    if raw_df.empty:
        return pd.DataFrame()

    df = raw_df.copy()
    source_name = source_path.stem

    video_name_col = find_first_column(df.columns, ["video_name", "video_id", "video", "source_filename"])
    video_path_col = find_first_column(df.columns, ["video_path", "storage_path", "source_path", "local_video_path"])
    camera_col = find_first_column(df.columns, ["camera_id", "camera", "camera_name"])
    track_col = find_first_column(df.columns, ["local_track_id", "track_id", "candidate_id", "human_key"])
    frame_col = find_first_column(df.columns, ["frame_idx", "frame", "frame_id"])
    timestamp_col = find_first_column(df.columns, ["timestamp", "time_utc", "captured_at", "created_at"])

    bbox_col = find_first_column(df.columns, ["bbox", "representative_bbox"])
    x1_col = find_first_column(df.columns, ["bbox_x1", "x1", "left"])
    y1_col = find_first_column(df.columns, ["bbox_y1", "y1", "top"])
    x2_col = find_first_column(df.columns, ["bbox_x2", "x2", "right"])
    y2_col = find_first_column(df.columns, ["bbox_y2", "y2", "bottom"])
    width_col = find_first_column(df.columns, ["bbox_w", "w", "width"])
    height_col = find_first_column(df.columns, ["bbox_h", "h", "height"])

    normalized_rows = []
    default_timestamp = pd.Timestamp(BATCH_TIMESTAMP_UTC)

    for _, row in df.iterrows():
        parsed_bbox = None
        if bbox_col is not None:
            parsed_bbox = parse_bbox_value(row.get(bbox_col))
        elif None not in (x1_col, y1_col):
            try:
                x1 = int(float(row.get(x1_col)))
                y1 = int(float(row.get(y1_col)))
                if x2_col is not None and y2_col is not None:
                    x2 = int(float(row.get(x2_col)))
                    y2 = int(float(row.get(y2_col)))
                elif width_col is not None and height_col is not None:
                    x2 = x1 + int(float(row.get(width_col)))
                    y2 = y1 + int(float(row.get(height_col)))
                else:
                    x2, y2 = None, None
                if None not in (x2, y2) and x2 > x1 and y2 > y1:
                    parsed_bbox = (x1, y1, x2, y2)
            except Exception:
                parsed_bbox = None

        if parsed_bbox is None:
            continue

        frame_idx = row.get(frame_col) if frame_col is not None else 0
        try:
            frame_idx = int(float(frame_idx))
        except Exception:
            frame_idx = 0

        timestamp_value = row.get(timestamp_col) if timestamp_col is not None else default_timestamp
        timestamp_value = pd.to_datetime(timestamp_value, utc=True, errors="coerce")
        if pd.isna(timestamp_value):
            timestamp_value = default_timestamp

        if video_path_col is not None and str(row.get(video_path_col) or "").strip():
            resolved_video_path = str(row.get(video_path_col)).strip()
        else:
            candidate_video_name = str(row.get(video_name_col) or source_name).strip() if video_name_col is not None else source_name
            resolved_video_path = str(Path(config["video_root_dir"]) / f"{candidate_video_name}.mp4")

        normalized_rows.append(
            {
                "camera_id": str(row.get(camera_col) or source_name).strip() if camera_col is not None else source_name,
                "video_name": str(row.get(video_name_col) or source_name).strip() if video_name_col is not None else source_name,
                "video_path": resolved_video_path,
                "local_track_id": str(row.get(track_col) or "").strip() if track_col is not None else "",
                "frame_idx": frame_idx,
                "x1": parsed_bbox[0],
                "y1": parsed_bbox[1],
                "x2": parsed_bbox[2],
                "y2": parsed_bbox[3],
                "source_timestamp_utc": timestamp_value,
                "source_file": source_path.name,
            }
        )

    normalized_df = pd.DataFrame(normalized_rows)
    if normalized_df.empty:
        return normalized_df

    normalized_df = normalized_df[normalized_df["local_track_id"].astype(str).str.len() > 0].copy()
    normalized_df.sort_values(["camera_id", "video_name", "local_track_id", "frame_idx"], inplace=True)
    normalized_df.reset_index(drop=True, inplace=True)
    return normalized_df


def load_tracking_batch(input_dir, config):
    input_path = Path(input_dir)
    frames = []
    for file_path in sorted(input_path.glob("*")):
        if file_path.suffix.lower() not in {".csv", ".parquet", ".json", ".jsonl"}:
            continue
        raw_df = load_rows_from_file(file_path)
        normalized_df = normalize_detection_frame(raw_df, file_path, config)
        if not normalized_df.empty:
            frames.append(normalized_df)
    if not frames:
        raise FileNotFoundError(f"No tracking files found in {input_dir}")
    batch_df = pd.concat(frames, ignore_index=True)
    batch_df.sort_values(["camera_id", "video_name", "local_track_id", "frame_idx"], inplace=True)
    batch_df.reset_index(drop=True, inplace=True)
    return batch_df


In [ ]:
def select_sample_indices(tracklet_size, max_samples):
    if tracklet_size <= max_samples:
        return np.arange(tracklet_size)
    return np.unique(np.linspace(0, tracklet_size - 1, num=max_samples, dtype=int))


def build_tracklet_table(detections_df, config):
    tracklet_rows = []
    grouped = detections_df.groupby(["camera_id", "video_name", "local_track_id"], sort=False)

    for (camera_id, video_name, local_track_id), group_df in grouped:
        if len(group_df) < config["min_frames_per_tracklet"]:
            continue

        sample_positions = select_sample_indices(len(group_df), config["samples_per_tracklet"])
        sampled_rows = group_df.iloc[sample_positions].copy()
        representative_row = group_df.iloc[len(group_df) // 2]

        tracklet_rows.append(
            {
                "camera_id": camera_id,
                "video_name": video_name,
                "video_path": str(group_df["video_path"].iloc[0]),
                "local_track_id": local_track_id,
                "source_timestamp_utc": pd.to_datetime(group_df["source_timestamp_utc"].min(), utc=True),
                "frame_count": int(len(group_df)),
                "start_frame_idx": int(group_df["frame_idx"].min()),
                "end_frame_idx": int(group_df["frame_idx"].max()),
                "representative_frame_idx": int(representative_row["frame_idx"]),
                "representative_bbox": [
                    int(representative_row["x1"]),
                    int(representative_row["y1"]),
                    int(representative_row["x2"]),
                    int(representative_row["y2"]),
                ],
                "sample_rows": sampled_rows[["frame_idx", "x1", "y1", "x2", "y2"]].to_dict("records"),
            }
        )

    tracklet_df = pd.DataFrame(tracklet_rows)
    if tracklet_df.empty:
        raise ValueError("No valid tracklets found after filtering")
    tracklet_df.reset_index(drop=True, inplace=True)
    return tracklet_df


detections_df = load_tracking_batch(CONFIG["tracking_input_dir"], CONFIG)
tracklet_df = build_tracklet_table(detections_df, CONFIG)

print("detections:", len(detections_df))
print("tracklets:", len(tracklet_df))
tracklet_df.head()


In [ ]:
def create_reid_extractor(config):
    return FeatureExtractor(
        model_name=config["reid_model_name"],
        device=config["device"],
    )


def clamp_bbox_to_frame(frame, bbox):
    frame_height, frame_width = frame.shape[:2]
    x1, y1, x2, y2 = [int(value) for value in bbox]
    x1 = max(0, min(x1, frame_width - 1))
    y1 = max(0, min(y1, frame_height - 1))
    x2 = max(0, min(x2, frame_width))
    y2 = max(0, min(y2, frame_height))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def crop_person_patch(frame, bbox, config):
    clamped_bbox = clamp_bbox_to_frame(frame, bbox)
    if clamped_bbox is None:
        return None
    x1, y1, x2, y2 = clamped_bbox
    if (x2 - x1) * (y2 - y1) < config["min_bbox_area"]:
        return None
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


def normalize_embedding(embedding_vector):
    embedding_vector = np.asarray(embedding_vector, dtype=np.float32).reshape(1, -1)
    faiss.normalize_L2(embedding_vector)
    return embedding_vector[0]


def extract_tracklet_embedding(tracklet_row, extractor, config):
    video_path = Path(tracklet_row["video_path"])
    if not video_path.exists():
        return None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return None

    embeddings = []
    for sample in tracklet_row["sample_rows"]:
        frame_idx = int(sample["frame_idx"])
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            continue

        crop = crop_person_patch(frame, [sample["x1"], sample["y1"], sample["x2"], sample["y2"]], config)
        if crop is None:
            continue

        feature_tensor = extractor([crop])
        feature_vector = feature_tensor.detach().cpu().numpy()[0]
        embeddings.append(feature_vector)

    cap.release()

    if not embeddings:
        return None

    mean_embedding = np.mean(np.vstack(embeddings).astype(np.float32), axis=0)
    return normalize_embedding(mean_embedding)


extractor = create_reid_extractor(CONFIG)
print("ReID extractor ready")


In [ ]:
def build_tracklet_embedding_table(tracklet_df, extractor, config):
    valid_rows = []
    valid_embeddings = []

    for _, tracklet_row in tqdm(tracklet_df.iterrows(), total=len(tracklet_df), desc="Extract embeddings"):
        embedding = extract_tracklet_embedding(tracklet_row, extractor, config)
        if embedding is None:
            continue
        valid_rows.append(tracklet_row.to_dict())
        valid_embeddings.append(embedding)

    if not valid_rows:
        raise ValueError("No embeddings were extracted from the current batch")

    valid_tracklet_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    embedding_matrix = np.vstack(valid_embeddings).astype(np.float32)
    return valid_tracklet_df, embedding_matrix


embedded_tracklet_df, embedded_tracklet_matrix = build_tracklet_embedding_table(tracklet_df, extractor, CONFIG)

print("embedded tracklets:", len(embedded_tracklet_df))
print("embedding matrix shape:", embedded_tracklet_matrix.shape)


In [ ]:
def load_json_state(state_path):
    if state_path.exists():
        return json.loads(state_path.read_text(encoding="utf-8"))
    return {
        "next_global_id": 1,
        "last_batch_id": None,
        "total_processed_tracklets": 0,
    }


def save_json_state(state_path, state_payload):
    state_path.write_text(json.dumps(state_payload, indent=2), encoding="utf-8")


def load_online_gallery(config):
    gallery_dir = Path(config["online_gallery_dir"])
    metadata_path = gallery_dir / "metadata.parquet"
    embedding_path = gallery_dir / "embeddings.npy"

    if metadata_path.exists() and embedding_path.exists():
        metadata_df = pd.read_parquet(metadata_path)
        embedding_matrix = np.load(embedding_path)
        if len(metadata_df) != len(embedding_matrix):
            raise ValueError("Online gallery metadata and embedding files are out of sync")
        if not metadata_df.empty:
            metadata_df["processed_at_utc"] = pd.to_datetime(metadata_df["processed_at_utc"], utc=True)
        return metadata_df.reset_index(drop=True), embedding_matrix.astype(np.float32)

    return pd.DataFrame(), np.empty((0, config["embedding_dim"]), dtype=np.float32)


def prune_online_gallery(metadata_df, embedding_matrix, config):
    if metadata_df.empty:
        return metadata_df, embedding_matrix

    cutoff = pd.Timestamp(BATCH_TIMESTAMP_UTC) - pd.Timedelta(hours=config["online_gallery_hours"])
    keep_mask = pd.to_datetime(metadata_df["processed_at_utc"], utc=True) >= cutoff
    pruned_metadata_df = metadata_df.loc[keep_mask].copy().reset_index(drop=True)
    pruned_embedding_matrix = embedding_matrix[keep_mask.to_numpy()]
    return pruned_metadata_df, pruned_embedding_matrix


def create_exact_faiss_index(embedding_matrix, config):
    index = faiss.IndexFlatIP(config["embedding_dim"])
    if len(embedding_matrix) > 0:
        index.add(embedding_matrix.astype(np.float32))
    return index


def assign_incremental_global_ids(tracklet_df, embedding_matrix, online_metadata_df, online_embedding_matrix, state_payload, config):
    current_online_metadata_df = online_metadata_df.copy()
    current_online_embedding_matrix = online_embedding_matrix.copy()
    current_index = create_exact_faiss_index(current_online_embedding_matrix, config)
    next_global_id = int(state_payload["next_global_id"])

    assigned_rows = []

    for row_index in tqdm(range(len(tracklet_df)), desc="Assign global IDs"):
        tracklet_row = tracklet_df.iloc[row_index]
        embedding_vector = embedding_matrix[row_index].reshape(1, -1).astype(np.float32)

        matched_global_id = None
        matched_score = None

        if current_index.ntotal > 0:
            distances, indices = current_index.search(embedding_vector, 1)
            matched_score = float(distances[0][0])
            matched_index = int(indices[0][0])
            if matched_index >= 0 and matched_score >= config["similarity_threshold"]:
                matched_global_id = int(current_online_metadata_df.iloc[matched_index]["global_id"])

        if matched_global_id is None:
            matched_global_id = next_global_id
            next_global_id += 1

        processed_row = {
            "batch_id": BATCH_ID,
            "processed_at_utc": pd.Timestamp(BATCH_TIMESTAMP_UTC),
            "global_id": matched_global_id,
            "camera_id": tracklet_row["camera_id"],
            "video_name": tracklet_row["video_name"],
            "video_path": tracklet_row["video_path"],
            "local_track_id": tracklet_row["local_track_id"],
            "source_timestamp_utc": pd.to_datetime(tracklet_row["source_timestamp_utc"], utc=True),
            "frame_count": int(tracklet_row["frame_count"]),
            "start_frame_idx": int(tracklet_row["start_frame_idx"]),
            "end_frame_idx": int(tracklet_row["end_frame_idx"]),
            "representative_frame_idx": int(tracklet_row["representative_frame_idx"]),
            "representative_bbox": tracklet_row["representative_bbox"],
            "sample_rows": tracklet_row["sample_rows"],
            "match_score": matched_score,
            "is_new_global_id": matched_score is None or matched_score < config["similarity_threshold"],
        }
        assigned_rows.append(processed_row)

        current_index.add(embedding_vector)
        current_online_embedding_matrix = np.vstack([current_online_embedding_matrix, embedding_vector])
        current_online_metadata_df = pd.concat(
            [current_online_metadata_df, pd.DataFrame([processed_row])],
            ignore_index=True,
        )

    assigned_df = pd.DataFrame(assigned_rows)
    state_payload["next_global_id"] = next_global_id
    state_payload["last_batch_id"] = BATCH_ID
    state_payload["total_processed_tracklets"] = int(state_payload.get("total_processed_tracklets", 0)) + len(assigned_df)
    return assigned_df, current_online_metadata_df, current_online_embedding_matrix, state_payload


def save_online_gallery(metadata_df, embedding_matrix, config):
    gallery_dir = Path(config["online_gallery_dir"])
    gallery_dir.mkdir(parents=True, exist_ok=True)
    metadata_df.to_parquet(gallery_dir / "metadata.parquet", index=False)
    np.save(gallery_dir / "embeddings.npy", embedding_matrix.astype(np.float32))


def save_partitioned_history(assigned_df, embedding_matrix, config):
    history_root = Path(config["history_dir"])
    batch_date = BATCH_TIMESTAMP_UTC.strftime("%Y-%m-%d")
    partition_dir = history_root / f"date={batch_date}"
    partition_dir.mkdir(parents=True, exist_ok=True)

    metadata_path = partition_dir / f"tracklets_{BATCH_ID}.parquet"
    embedding_path = partition_dir / f"embeddings_{BATCH_ID}.npy"

    assigned_df.to_parquet(metadata_path, index=False)
    np.save(embedding_path, embedding_matrix.astype(np.float32))
    return metadata_path, embedding_path


def save_batch_output(assigned_df, config):
    output_dir = Path(config["batch_output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"unified_tracklets_{BATCH_ID}.parquet"
    assigned_df.to_parquet(output_path, index=False)
    return output_path


In [ ]:
state_path = Path(CONFIG["working_dir"]) / "state.json"
state_payload = load_json_state(state_path)

online_metadata_df, online_embedding_matrix = load_online_gallery(CONFIG)
online_metadata_df, online_embedding_matrix = prune_online_gallery(online_metadata_df, online_embedding_matrix, CONFIG)

assigned_df, updated_online_metadata_df, updated_online_embedding_matrix, updated_state_payload = assign_incremental_global_ids(
    embedded_tracklet_df,
    embedded_tracklet_matrix,
    online_metadata_df,
    online_embedding_matrix,
    state_payload,
    CONFIG,
)

batch_output_path = save_batch_output(assigned_df, CONFIG)
history_metadata_path, history_embedding_path = save_partitioned_history(assigned_df, embedded_tracklet_matrix, CONFIG)
save_online_gallery(updated_online_metadata_df, updated_online_embedding_matrix, CONFIG)
save_json_state(state_path, updated_state_payload)

summary = {
    "batch_id": BATCH_ID,
    "processed_tracklets": int(len(assigned_df)),
    "new_global_ids": int(assigned_df["is_new_global_id"].sum()),
    "existing_global_id_matches": int((~assigned_df["is_new_global_id"]).sum()),
    "online_gallery_size": int(len(updated_online_metadata_df)),
    "next_global_id": int(updated_state_payload["next_global_id"]),
    "batch_output_path": str(batch_output_path),
    "history_metadata_path": str(history_metadata_path),
    "history_embedding_path": str(history_embedding_path),
}

print(json.dumps(summary, indent=2, default=str))
assigned_df.head()


## Query Artifact Build

The next cells add both query directions into the same notebook:

- `vector retrieval`: search directly on ReID embeddings.
- `Qwen description/schema`: generate text descriptions and structured metadata from representative crops.
- `hybrid retrieval`: combine vector, text, and schema filters.

Set `CONFIG["enable_qwen_annotation"] = True` if you want the notebook to actually download and run the Qwen VLM on Kaggle.


In [ ]:
def ensure_directory(path_value):
    path = Path(path_value)
    path.mkdir(parents=True, exist_ok=True)
    return path


def extract_representative_image(tracklet_row, output_dir, config):
    output_dir = ensure_directory(output_dir)
    output_path = output_dir / f"{BATCH_ID}_gid_{int(tracklet_row['global_id'])}_cam_{tracklet_row['camera_id']}_trk_{tracklet_row['local_track_id']}.png"
    if output_path.exists():
        return output_path

    video_path = Path(tracklet_row["video_path"])
    if not video_path.exists():
        return None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return None

    cap.set(cv2.CAP_PROP_POS_FRAMES, int(tracklet_row["representative_frame_idx"]))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        return None

    crop = crop_person_patch(frame, tracklet_row["representative_bbox"], config)
    if crop is None:
        return None

    Image.fromarray(crop).save(output_path)
    return output_path


def infer_basic_color_name(rgb_vector):
    r, g, b = [float(value) for value in rgb_vector]
    brightness = (r + g + b) / 3.0
    if brightness < 45:
        return "black"
    if brightness > 210:
        return "white"
    if abs(r - g) < 20 and abs(g - b) < 20:
        return "gray"
    dominant = np.argmax([r, g, b])
    if dominant == 0:
        return "red"
    if dominant == 1:
        return "green"
    return "blue"


def extract_basic_visual_schema(image_path):
    if image_path is None or not Path(image_path).exists():
        return {}

    image = np.asarray(Image.open(image_path).convert("RGB"))
    height = image.shape[0]
    upper_half = image[: max(1, height // 2), :, :]
    lower_half = image[max(1, height // 2):, :, :]

    upper_color = infer_basic_color_name(np.mean(upper_half.reshape(-1, 3), axis=0))
    lower_color = infer_basic_color_name(np.mean(lower_half.reshape(-1, 3), axis=0))
    aspect_ratio = round(float(image.shape[0]) / max(float(image.shape[1]), 1.0), 3)

    return {
        "upper_color_hint": upper_color,
        "lower_color_hint": lower_color,
        "crop_height": int(image.shape[0]),
        "crop_width": int(image.shape[1]),
        "crop_aspect_ratio": aspect_ratio,
    }


def build_query_base_artifacts(assigned_df, reid_embedding_matrix, config):
    query_root = ensure_directory(config["query_artifact_dir"])
    image_dir = ensure_directory(query_root / "representative_images")

    query_rows = []
    for row_index in tqdm(range(len(assigned_df)), desc="Build query artifacts"):
        tracklet_row = assigned_df.iloc[row_index].to_dict()
        image_path = extract_representative_image(tracklet_row, image_dir, config)
        visual_schema = extract_basic_visual_schema(image_path)

        query_rows.append(
            {
                **tracklet_row,
                "reid_embedding_index": row_index,
                "representative_image_path": str(image_path) if image_path is not None else None,
                "basic_visual_schema": visual_schema,
                "description_text": None,
                "schema_json": visual_schema,
                "query_text_corpus": "",
            }
        )

    query_df = pd.DataFrame(query_rows)
    query_df["processed_at_utc"] = pd.to_datetime(query_df["processed_at_utc"], utc=True)
    query_df["source_timestamp_utc"] = pd.to_datetime(query_df["source_timestamp_utc"], utc=True)

    query_df.to_parquet(query_root / "query_base.parquet", index=False)
    np.save(query_root / "reid_embeddings.npy", reid_embedding_matrix.astype(np.float32))
    return query_df


query_df = build_query_base_artifacts(assigned_df, embedded_tracklet_matrix, CONFIG)
print("query rows:", len(query_df))
query_df[["global_id", "camera_id", "video_name", "representative_image_path"]].head()


In [ ]:
def extract_json_block(raw_text):
    if raw_text is None:
        return None
    text = str(raw_text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match is None:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None


def flatten_schema_to_text(schema_obj):
    if not isinstance(schema_obj, dict):
        return ""
    parts = []
    for key, value in schema_obj.items():
        if value in [None, "", [], {}]:
            continue
        if isinstance(value, list):
            value = ", ".join(str(item) for item in value)
        parts.append(f"{key}: {value}")
    return " | ".join(parts)


def load_qwen_pipeline(config):
    from transformers import pipeline

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    return pipeline(
        task="image-text-to-text",
        model=config["qwen_vlm_model_name"],
        device_map="auto",
        torch_dtype=dtype,
    )


def qwen_describe_tracklet_image(image_path, qwen_pipe):
    prompt = (
        "You are given one cropped image of a person tracklet. "
        "Return only valid JSON with two top-level keys: "
        "description and schema. "
        "description must be a short retrieval-friendly sentence. "
        "schema must be an object with keys: upper_color, lower_color, bag, hat, gender_hint, age_group_hint, "
        "sleeve_length, lower_length, footwear_color, accessories, confidence. "
        "Use null when unknown. accessories must be a list. confidence must be between 0 and 1."
    )

    image = Image.open(image_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    response = qwen_pipe(text=messages, max_new_tokens=256)
    if not response:
        return None, {}

    raw_generated = response[0].get("generated_text")
    if isinstance(raw_generated, list) and raw_generated:
        last_item = raw_generated[-1]
        if isinstance(last_item, dict):
            raw_generated = last_item.get("content", last_item)

    parsed = extract_json_block(raw_generated)
    if not isinstance(parsed, dict):
        return str(raw_generated), {}

    description = str(parsed.get("description") or "").strip()
    schema = parsed.get("schema") if isinstance(parsed.get("schema"), dict) else {}
    return description, schema


def annotate_query_df_with_qwen(query_df, config):
    working_df = query_df.copy()
    if not config["enable_qwen_annotation"]:
        working_df["description_text"] = working_df["description_text"].fillna("")
        working_df["query_text_corpus"] = working_df["schema_json"].apply(flatten_schema_to_text)
        return working_df

    qwen_pipe = load_qwen_pipeline(config)
    max_rows = min(len(working_df), int(config["qwen_max_tracklets"]))

    for row_index in tqdm(range(max_rows), desc="Qwen annotation"):
        image_path = working_df.iloc[row_index]["representative_image_path"]
        if not image_path:
            continue
        description, qwen_schema = qwen_describe_tracklet_image(image_path, qwen_pipe)
        merged_schema = dict(working_df.iloc[row_index]["basic_visual_schema"] or {})
        merged_schema.update(qwen_schema or {})
        working_df.at[row_index, "description_text"] = description
        working_df.at[row_index, "schema_json"] = merged_schema

    working_df["description_text"] = working_df["description_text"].fillna("")
    working_df["query_text_corpus"] = working_df.apply(
        lambda row: " | ".join(
            part for part in [
                str(row.get("description_text") or "").strip(),
                flatten_schema_to_text(row.get("schema_json") or {}),
                f"camera_id: {row.get('camera_id')}",
                f"video_name: {row.get('video_name')}",
            ]
            if part
        ),
        axis=1,
    )
    return working_df


def load_text_embedding_model(config):
    from sentence_transformers import SentenceTransformer

    return SentenceTransformer(config["text_embedding_model_name"], device=config["device"])


def encode_text_corpus(text_list, text_model):
    if not text_list:
        return np.empty((0, 384), dtype=np.float32)
    embeddings = text_model.encode(
        text_list,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    return embeddings.astype(np.float32)


In [ ]:
annotated_query_df = annotate_query_df_with_qwen(query_df, CONFIG)

text_embedding_matrix = None
if annotated_query_df["query_text_corpus"].fillna("").str.len().gt(0).any():
    text_model = load_text_embedding_model(CONFIG)
    text_embedding_matrix = encode_text_corpus(annotated_query_df["query_text_corpus"].fillna("").tolist(), text_model)

query_root = ensure_directory(CONFIG["query_artifact_dir"])
annotated_query_df.to_parquet(query_root / "query_index.parquet", index=False)
np.save(query_root / "reid_embeddings.npy", embedded_tracklet_matrix.astype(np.float32))
if text_embedding_matrix is not None:
    np.save(query_root / "text_embeddings.npy", text_embedding_matrix.astype(np.float32))

query_summary = {
    "query_index_rows": int(len(annotated_query_df)),
    "qwen_enabled": bool(CONFIG["enable_qwen_annotation"]),
    "text_embeddings_ready": bool(text_embedding_matrix is not None),
    "query_index_path": str(query_root / "query_index.parquet"),
}
print(json.dumps(query_summary, indent=2))
annotated_query_df[["global_id", "description_text", "schema_json", "query_text_corpus"]].head()


In [ ]:
def safe_float(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return float(default)


def load_query_artifacts(config):
    query_root = Path(config["query_artifact_dir"])
    metadata_df = pd.read_parquet(query_root / "query_index.parquet")
    reid_embeddings = np.load(query_root / "reid_embeddings.npy")

    text_embedding_path = query_root / "text_embeddings.npy"
    text_embeddings = np.load(text_embedding_path) if text_embedding_path.exists() else None

    metadata_df["processed_at_utc"] = pd.to_datetime(metadata_df["processed_at_utc"], utc=True)
    metadata_df["source_timestamp_utc"] = pd.to_datetime(metadata_df["source_timestamp_utc"], utc=True)
    return metadata_df, reid_embeddings.astype(np.float32), None if text_embeddings is None else text_embeddings.astype(np.float32)


def build_faiss_index_from_embeddings(embedding_matrix):
    if embedding_matrix is None or len(embedding_matrix) == 0:
        return None
    index = faiss.IndexFlatIP(int(embedding_matrix.shape[1]))
    index.add(embedding_matrix.astype(np.float32))
    return index


def query_by_reference_global_id(reference_global_id, metadata_df, reid_embeddings, top_k=10):
    matches = metadata_df.index[metadata_df["global_id"].astype(int) == int(reference_global_id)].tolist()
    if not matches:
        raise ValueError(f"global_id not found: {reference_global_id}")

    anchor_index = int(matches[0])
    index = build_faiss_index_from_embeddings(reid_embeddings)
    scores, indices = index.search(reid_embeddings[anchor_index].reshape(1, -1).astype(np.float32), min(top_k, len(metadata_df)))

    result_df = metadata_df.iloc[indices[0]].copy().reset_index(drop=True)
    result_df["reid_score"] = scores[0]
    return result_df


def query_by_text_description(query_text, metadata_df, text_embeddings, config, top_k=10):
    if text_embeddings is None or len(text_embeddings) == 0:
        raise ValueError("Text embeddings are not available. Enable Qwen annotation or build text corpus first.")

    text_model = load_text_embedding_model(config)
    query_vector = text_model.encode([query_text], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    index = build_faiss_index_from_embeddings(text_embeddings)
    scores, indices = index.search(query_vector, min(top_k, len(metadata_df)))

    result_df = metadata_df.iloc[indices[0]].copy().reset_index(drop=True)
    result_df["text_score"] = scores[0]
    return result_df


def filter_metadata_df(metadata_df, schema_filters):
    if not schema_filters:
        return metadata_df.copy()

    keep_rows = []
    for _, row in metadata_df.iterrows():
        schema = row.get("schema_json") or {}
        is_match = True
        for key, expected_value in schema_filters.items():
            actual_value = schema.get(key)
            if isinstance(expected_value, list):
                actual_list = actual_value if isinstance(actual_value, list) else [actual_value]
                if not any(str(item).lower() == str(candidate).lower() for item in actual_list for candidate in expected_value):
                    is_match = False
                    break
            else:
                if str(actual_value).lower() != str(expected_value).lower():
                    is_match = False
                    break
        keep_rows.append(is_match)

    return metadata_df.loc[np.asarray(keep_rows, dtype=bool)].copy().reset_index(drop=True)


def hybrid_query(query_text, metadata_df, reid_embeddings, text_embeddings, config, anchor_global_id=None, schema_filters=None, top_k=10):
    working_df = metadata_df.copy().reset_index(drop=True)
    working_df["hybrid_score"] = 0.0

    if schema_filters:
        filtered_df = filter_metadata_df(working_df, schema_filters)
        filtered_ids = set(filtered_df["global_id"].astype(int).tolist())
        working_df["schema_score"] = working_df["global_id"].astype(int).apply(lambda gid: 1.0 if gid in filtered_ids else 0.0)
        working_df["hybrid_score"] += 0.20 * working_df["schema_score"]
    else:
        working_df["schema_score"] = 0.0

    if query_text and text_embeddings is not None and len(text_embeddings) == len(working_df):
        text_model = load_text_embedding_model(config)
        query_vector = text_model.encode([query_text], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
        text_scores = np.dot(text_embeddings, query_vector[0])
        working_df["text_score"] = text_scores
        working_df["hybrid_score"] += 0.35 * working_df["text_score"]
    else:
        working_df["text_score"] = 0.0

    if anchor_global_id is not None:
        anchor_matches = working_df.index[working_df["global_id"].astype(int) == int(anchor_global_id)].tolist()
        if anchor_matches:
            anchor_embedding = reid_embeddings[int(anchor_matches[0])]
            reid_scores = np.dot(reid_embeddings, anchor_embedding)
            working_df["reid_score"] = reid_scores
            working_df["hybrid_score"] += 0.45 * working_df["reid_score"]
        else:
            working_df["reid_score"] = 0.0
    else:
        working_df["reid_score"] = 0.0

    return working_df.sort_values("hybrid_score", ascending=False).head(top_k).reset_index(drop=True)


query_metadata_df, query_reid_embeddings, query_text_embeddings = load_query_artifacts(CONFIG)
print("loaded query artifacts:", len(query_metadata_df))


In [ ]:
# Example experiments. Change these values and rerun this cell.
REFERENCE_GLOBAL_ID = int(query_metadata_df.iloc[0]["global_id"]) if len(query_metadata_df) > 0 else None
QUERY_TEXT = "person with blue upper clothes and dark lower clothes"
SCHEMA_FILTERS = {"upper_color": "blue"}

vector_results_df = None
text_results_df = None
hybrid_results_df = None

if REFERENCE_GLOBAL_ID is not None:
    vector_results_df = query_by_reference_global_id(
        reference_global_id=REFERENCE_GLOBAL_ID,
        metadata_df=query_metadata_df,
        reid_embeddings=query_reid_embeddings,
        top_k=CONFIG["query_top_k"],
    )

try:
    text_results_df = query_by_text_description(
        query_text=QUERY_TEXT,
        metadata_df=query_metadata_df,
        text_embeddings=query_text_embeddings,
        config=CONFIG,
        top_k=CONFIG["query_top_k"],
    )
except Exception as exc:
    print("text query skipped:", exc)

hybrid_results_df = hybrid_query(
    query_text=QUERY_TEXT,
    metadata_df=query_metadata_df,
    reid_embeddings=query_reid_embeddings,
    text_embeddings=query_text_embeddings,
    config=CONFIG,
    anchor_global_id=REFERENCE_GLOBAL_ID,
    schema_filters=SCHEMA_FILTERS,
    top_k=CONFIG["query_top_k"],
)

print("vector results")
display_columns = ["global_id", "camera_id", "video_name", "description_text", "schema_json"]
if vector_results_df is not None:
    display(vector_results_df[display_columns + ["reid_score"]].head())

print("text results")
if text_results_df is not None:
    display(text_results_df[display_columns + ["text_score"]].head())

print("hybrid results")
display(hybrid_results_df[display_columns + ["schema_score", "text_score", "reid_score", "hybrid_score"]].head())


## Practical notes

- You do not need to re-process old videos every 10 minutes.
- You only extract embeddings for the new batch.
- You only search against the persisted online gallery.
- The online gallery is pruned by time window to keep cost bounded.
- Full history stays partitioned by date for later analytics or offline retrieval.
- After this notebook finishes, the raw queue videos for the processed batch can be deleted safely if your retention policy allows it.
- `Vector retrieval` is usually the strongest path for identity matching.
- `Qwen description/schema` is useful for text search and filtering, but should normally support the vector path rather than replace it.

Recommended evaluation order:

1. Test `query_by_reference_global_id` first to validate ReID consistency.
2. Then enable Qwen and inspect `description_text` plus `schema_json` quality.
3. Finally compare `text_results_df` and `hybrid_results_df` on your real queries.
